# Overview of Data exploration

---

Here I am exploring the Olivetti dataset and checking what I will be working with.

What the dataset gives me:

```
dataset.images = (400, 64, 64)   # 400 faces, 64x64 grid
dataset.data   = (400, 4096)     # each face flattened (4096 = 64 * 64)
dataset.target = (400,)          # 0-39, ten photos per person
```

The pipeline code now lives in the `fem/` package (`fem.data`, `fem.maps`, `fem.metrics`), so this
notebook does not duplicate it — it only looks at the data and at Map A's numbers. All measured
numbers come from `analysis/run_olivetti.py` and are stored in `results/olivetti.json`; the written
findings are in `findings.md`.

In [ ]:
import sys
from pathlib import Path

# The notebook lives in "Map A - Raw Pixels/"; the fem package is one level up.
ROOT = Path.cwd().resolve().parent
if (ROOT / "fem").is_dir():
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from fem.data import load_olivetti
from fem.maps import eigenfaces, pca_map, rank_correlation_2d, variance_milestones
from fem.metrics import knn_loo, silhouette

In [ ]:
images, labels = load_olivetti()

print(f"images: {images.shape}  dtype={images.dtype}")
print(f"labels: {labels.shape}  identities={np.unique(labels).size}")
print(f"images per identity: {np.bincount(labels).tolist()}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image in zip(axes.ravel(), images[:10]):
    ax.imshow(image, cmap="gray")
    ax.axis("off")
fig.suptitle("First 10 Olivetti faces (all person 0)")
plt.show()

In [ ]:
# The dataset spans the full scale, but any single image covers a narrower band.
print(f"dataset-wide pixel range: [{images.min():.4f}, {images.max():.4f}]")
print(f"image 0 pixel range:      [{images[0].min():.4f}, {images[0].max():.4f}]")
print(f"first 20 flattened values of image 0: {images[0].ravel()[:20]}")

# Map A: PCA on raw pixels

## What the components look like

**Component 0** is a broad brightness gradient across the face; **component 1** is bright on the
left and dark on the right. Both encode illumination direction, not identity.

![First 16 eigenfaces](../figures/eigenfaces.png)

## How much variance the components hold

Component 0 alone explains 23.81% of the variance. It takes **2** components to reach 25%, **4** to
reach 50%, **19** to reach 75% and **66** to reach 90%. 95% is **not** reached within the first 100
components. The cumulative curve (Map A blue, Map B red) is in the two-map scree plot:

![Scree and cumulative variance](../figures/scree.png)

**Eyeballing failed twice:** my first guess from the component images (a handful mattered) and my
revised guess from the scree plot were both low. The flat tail is not empty — components 20-66 hold
15% of the variance between them.

**Decision:** no StandardScaler. All 4096 pixel columns share one unit (brightness 0-1), so
per-column scaling just amplifies near-constant background. Classic eigenfaces centres but does not
scale.

In [ ]:
X = images.reshape(len(images), -1)  # (400, 4096) raw pixel vectors

pca100 = pca_map(X, n_components=100, seed=0)
evr = pca100.evr

print(f"component 0: {evr[0] * 100:.4f}% of variance")
print(f"variance milestones (components needed): {variance_milestones(evr)}")

In [ ]:
faces = eigenfaces(pca100.model.components_, n=16)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for index, (ax, face) in enumerate(zip(axes.ravel(), faces), start=1):
    limit = np.abs(face).max()
    ax.imshow(face, cmap="gray", vmin=-limit, vmax=limit)
    ax.set_title(f"PC {index}", fontsize=8)
    ax.axis("off")
fig.suptitle("First 16 eigenfaces")
plt.show()

In [ ]:
pca2 = pca_map(X, n_components=2, seed=0)
coords = pca2.coords

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(coords[:, 0], coords[:, 1], c=labels, cmap="tab20", s=14, alpha=0.85)
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_title("Map A: raw pixels, PCA to 2-D (colour = identity)")
plt.show()

print(f"2-D vs full-dimensional geometry (Spearman): {rank_correlation_2d(coords, X):.4f}")

## The 2-D scatter is for eyes only

The picture above is what the project started with. It is **not** a measurement: its Spearman
correlation with the full-dimensional pairwise distances is 0.8070 for Map A (0.5174 for Map B), so
it preserves only part of the true geometry. Every metric in the findings and the paper is computed
on the 4096-d vectors, never on these 2-D coordinates. The figure used in the write-up is
`figures/mapA_scatter.png`, regenerated by `analysis/run_olivetti.py`.

![Map A 2-D PCA scatter](../figures/mapA_scatter.png)

In [ ]:
print(f"silhouette (full 4096-d): {silhouette(X, labels):.5f}")

knn = knn_loo(X, labels, k=1)
print(f"1-NN leave-one-out accuracy: {knn['accuracy']:.4f}")
print(f"top-5 accuracy:              {knn['top5']:.4f}")

# What these numbers say

Map A clusters weakly and identifies surprisingly well: silhouette 0.10557, 1-NN LOO 0.9450. But
verification on the same vectors fails badly (ROC-AUC 0.88071, EER 0.2050), and the leading
components are illumination, not identity. Map B, under identical PCA and metric settings, reaches
silhouette 0.49698 and EER 0.011111 — the side-by-side numbers live in `results/olivetti.json`, the
comparison in the paper, and the prose in `findings.md`.

Two traps this notebook is careful about:

1. **The 2-D plot is never measured.** It is for human eyes; metrics use the full vector.
2. **Nothing here trains.** PCA is fit by `fem.maps.pca_map` on centred raw pixels only, with the
   same settings used for Map B (`n_components=2`, `svd_solver="randomized"`, `random_state=0`).